# 1. Graph 분석

In [11]:
import os
import sys
sys.path.append(os.path.abspath('../src'))

import re
import ast
import json
import networkx as nx

from utils.graph_visualizer import GraphVisualizer
from modules.builders.graph_builder import HeteroGraphBuilder

In [12]:
visualizer = GraphVisualizer(output_dir="./logs/visualizations")
graph_builder = HeteroGraphBuilder(db_dir="./data/raw/BIRD_dev/dev_databases") # DB 경로

In [13]:
def graph_visualizer(log_file_path, target_query_id, exp_name):
    dev_json_path = "/home/hyeonjin/thesis_refactored/data/raw/BIRD_dev/dev.json"

    parsed_data = {
        'question': '', 'metadata': {}, 'seeds': [], 'final_nodes': [], 
        'node_scores': [], 'generated_sql': '', 'gold_schema': []
    }

    # ==========================================================
    # 3. .log 파일 파싱 (질문, 메타데이터, 예측 노드, 예측 SQL 추출)
    # ==========================================================
    is_target_query = False
    in_metadata_block = False
    in_sql_block = False
    metadata_str = ""

    question_start_pattern = re.compile(rf"Question\s+{target_query_id}:")
    question_any_pattern = re.compile(r"Question\s+\d+:")
    log_time_pattern = re.compile(r"^\[20\d{2}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\]")

    try:
        with open(log_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if question_start_pattern.search(line):
                    is_target_query = True
                    parsed_data['question'] = line.split(f"Question {target_query_id}:")[-1].strip()
                    continue
                    
                if is_target_query and question_any_pattern.search(line) and not question_start_pattern.search(line):
                    break
                if not is_target_query:
                    continue

                if in_sql_block:
                    if log_time_pattern.match(line):
                        in_sql_block = False # 새로운 로그 줄이 시작되면 캡처 종료
                    else:
                        # 줄바꿈을 유지하면서 SQL 이어 붙이기
                        parsed_data['generated_sql'] += "\n" + line.strip()
                        continue

                if "metadata: {" in line:
                    metadata_str = line.split("metadata: ")[1].strip()
                    if metadata_str.count('{') == metadata_str.count('}') and metadata_str.count('[') == metadata_str.count(']'):
                        try: parsed_data['metadata'] = ast.literal_eval(metadata_str)
                        except: pass
                    else:
                        in_metadata_block = True
                elif in_metadata_block:
                    metadata_str += line.strip()
                    if metadata_str.count('{') == metadata_str.count('}') and metadata_str.count('[') == metadata_str.count(']'):
                        in_metadata_block = False
                        try: parsed_data['metadata'] = ast.literal_eval(metadata_str)
                        except: pass
                
                if "node_scores: tensor([" in line:
                    score_str = line.split("tensor([")[1].split("])")[0].replace('\n', '').strip()
                    try: parsed_data['node_scores'] = [float(x) for x in score_str.split(',') if x.strip()]
                    except: pass
                    
                if "seeds: [" in line:
                    try: parsed_data['seeds'] = ast.literal_eval(line.split("seeds: ")[1].strip())
                    except: pass
                    
                if "Final Nodes: [" in line:
                    try: parsed_data['final_nodes'] = ast.literal_eval(line.split("Final Nodes: ")[1].strip())
                    except: pass
                    
                if "Generated SQL:" in line:
                    parsed_data['generated_sql'] = line.split("Generated SQL:")[1].strip()
                    in_sql_block = True
                    
    except FileNotFoundError:
        print(f"❌ 로그 파일을 찾을 수 없습니다: {log_file_path}")

    # ==========================================================
    # 4. dev.json 연동 및 Gold Nodes(정답) 추출
    # ==========================================================
    gold_sql = ""
    try:
        with open(dev_json_path, 'r', encoding='utf-8') as f:
            dev_data = json.load(f)
            
        # 질문 텍스트로 1차 매칭, 실패 시 ID로 2차 매칭
        for item in dev_data:
            if item.get('question', '').strip() == parsed_data['question']:
                gold_sql = item.get('SQL', item.get('query', ''))
                break
        if not gold_sql and target_query_id < len(dev_data):
            gold_sql = dev_data[target_query_id].get('SQL', dev_data[target_query_id].get('query', ''))
    except FileNotFoundError:
        print(f"⚠️ dev.json 파일을 찾을 수 없습니다: {dev_json_path}")

    node_meta = parsed_data.get('metadata', {}).get('node_metadata', {})
    if gold_sql and node_meta:
        gold_sql_lower = gold_sql.lower()
        for idx, name in node_meta.items():
            if "->" in name: continue 
            search_target = name.split(".", 1)[1] if "." in name else name
            pattern = rf'\b{re.escape(search_target.lower())}\b'
            if re.search(pattern, gold_sql_lower):
                parsed_data['gold_schema'].append(name)

    # ==========================================================
    # 5. NetworkX 그래프 재건 및 텍스트 변환
    # ==========================================================
    nx_graph = nx.Graph()
    edges = parsed_data.get('metadata', {}).get('edges', [])
    edge_types = parsed_data.get('metadata', {}).get('edge_types', [])
    scores = parsed_data['node_scores']

    if node_meta:
        for idx, name in node_meta.items():
            idx = int(idx)
            n_type = "table" if "." not in name and "->" not in name else "column"
            score = scores[idx] if idx < len(scores) else 0.0
            nx_graph.add_node(name, name=name, type=n_type, similarity_score=round(score, 4))
            
        for i, (u_idx, v_idx) in enumerate(edges):
            u_name, v_name = node_meta.get(u_idx), node_meta.get(v_idx)
            if u_name and v_name:
                nx_graph.add_edge(u_name, v_name, type=edge_types[i] if i < len(edge_types) else "relation")

    seeds_text = [node_meta.get(idx, str(idx)) for idx in parsed_data['seeds']] if parsed_data['seeds'] else []

    # ==========================================================
    # 6. 📊 깔끔한 종합 진단 리포트 출력 (여기가 핵심!)
    # ==========================================================
    print("="*60)
    print(f"✅ 파싱 및 진단 완료: Question {target_query_id}")
    print("="*60)
    print(f"📝 Question     : {parsed_data['question']}")
    print(f"🎯 Gold SQL     : {gold_sql}")
    print(f"🤖 Generated SQL: {parsed_data['generated_sql']}")
    print("-" * 60)
    print(f"📊 Graph Size   : {nx_graph.number_of_nodes()} Nodes / {nx_graph.number_of_edges()} Edges")
    print(f"🏆 Gold Nodes   ({len(parsed_data['gold_schema'])}개): {parsed_data['gold_schema']}")
    print(f"🌱 Seed Nodes   ({len(seeds_text)}개): {seeds_text}")
    print(f"🏁 Final Nodes  ({len(parsed_data['final_nodes'])}개): {parsed_data['final_nodes']}")
    print("="*60)

    # ==========================================================
    # 7. HTML 렌더링
    # ==========================================================
    if nx_graph.number_of_nodes() > 0:
        html_file_name = f"analysis_{exp_name}_q{target_query_id}.html"
        visualizer.visualize(
            graph=nx_graph,
            question=parsed_data['question'],
            seeds=seeds_text,
            final_nodes=parsed_data['final_nodes'],
            gold_nodes=parsed_data['gold_schema'], 
            file_name=html_file_name
        )
        print(f"HTML 파일이 {html_file_name}으로 저장되었습니다.")
    else:
        print("❌ 그래프 메타데이터가 파싱되지 않아 시각화를 건너뜁니다.")

In [16]:
log_file_path = "/home/hyeonjin/thesis_refactored/logs/baselines/baseline_g_retriever/baseline_g_retriever_20260328_022118.log"
target_query_id = 8
exp_name = "baseline_g_retriever"

In [17]:
graph_visualizer(log_file_path, target_query_id, exp_name)

✅ 파싱 및 진단 완료: Question 8
📝 Question     : What is the number of SAT test takers of the schools with the highest FRPM count for K-12 students?
🎯 Gold SQL     : SELECT NumTstTakr FROM satscores WHERE cds = ( SELECT CDSCode FROM frpm ORDER BY `FRPM Count (K-12)` DESC LIMIT 1 )
🤖 Generated SQL: SELECT satscores.NumTstTakr
FROM satscores
JOIN frpm ON satscores.cds = frpm.CDSCode
WHERE frpm."FRPM Count (K-12)" = (
SELECT MAX("FRPM Count (K-12)")
FROM frpm
)
------------------------------------------------------------
📊 Graph Size   : 94 Nodes / 93 Edges
🏆 Gold Nodes   (6개): ['frpm', 'satscores', 'frpm.CDSCode', 'satscores.cds', 'satscores.NumTstTakr', 'schools.CDSCode']
🌱 Seed Nodes   (94개): ['frpm.FRPM Count (K-12)', 'frpm.Enrollment (K-12)', 'schools', 'frpm.FRPM Count (Ages 5-17)', 'frpm.Enrollment (Ages 5-17)', 'frpm.Free Meal Count (K-12)', 'frpm.Charter School (Y/N)', 'frpm.Percent (%) Eligible FRPM (K-12)', 'frpm.Charter School Number', 'frpm.2013-14 CALPADS Fall 1 Certification Statu